In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Load feature_df from CSV
feature_df_path = "../data/feature_df.csv"
feature_df = pd.read_csv(feature_df_path, index_col=0)
print(f"✅ Loaded feature_df from {feature_df_path}")
print(f"   Shape: {feature_df.shape}")
print(f"   Features: {list(feature_df.columns)}")
print()


✅ Loaded feature_df from ../data/feature_df.csv
   Shape: (26835, 5)
   Features: ['mean_price_usd', 'mean_pct_change', 'volatility', 'mean_deviation', 'spike_count']



In [2]:
# Linear Predictor
def train_risk_model(feature_df):
    """
    feature_df: per-skin aggregated features.
    Must contain: mean_price_usd, mean_pct_change, volatility, mean_deviation, spike_count
    Returns: trained LDA model, scaler, and the cleaned feature_df with risky_label.
    """

    # --- 0. Work on a copy so we don't mutate the caller's dataframe ---
    feature_df = feature_df.copy()

    # --- 1. Create pseudo-label (risky_label) based on extremes ---
    spike_threshold = feature_df["spike_count"].quantile(0.9)  # top 10% spikes
    vol_threshold   = feature_df["volatility"].quantile(0.9)   # top 10% volatility

    feature_df["risky_label"] = np.where(
        (feature_df["spike_count"] >= spike_threshold) |
        (feature_df["volatility"]   >= vol_threshold),
        1,
        0
    )

    # --- 2. Build X matrix from the selected features ---
    X = feature_df[[
        "mean_price_usd",
        "mean_pct_change",
        "volatility",
        "mean_deviation",
        "spike_count"
    ]].copy()

    y = feature_df["risky_label"].values

    # --- 3. Clean X: handle inf, NaN, giant values ---

    # Replace +/- inf with NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # Fill NaN with 0 (you could also choose column medians if you prefer)
    X = X.fillna(0)

    # Clip insane outliers so scaling doesn't explode.
    # This makes sure a single crazy number (like 1e12) doesn't dominate MinMax.
    X = np.clip(X, -1e6, 1e6)

    # At this point X is a NumPy array again
    X = np.array(X, dtype=float)
    

    # --- 4. Train/test split ---
    # It's possible all risky_label are 0 or 1 if data is super uniform,
    # so we wrap stratify in a try/except for robustness.
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=0, stratify=y
        )
    except ValueError:
        # stratify can fail if there's only one class. fallback:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=0
        )

    # --- 5. Scale features 0-1 ---
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # --- 6. Train LDA ---
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_train_scaled, y_train)

    # --- 7. Evaluate ---
    train_acc = lda.score(X_train_scaled, y_train)
    test_acc  = lda.score(X_test_scaled,  y_test)

    print(f"[LDA] Training accuracy: {train_acc:.3f}")
    print(f"[LDA] Testing  accuracy: {test_acc:.3f}")
    print("Class balance (0 normal / 1 risky):")
    print(pd.Series(y).value_counts())

    return lda, scaler, feature_df


In [3]:
def generate_client_report(feature_df, lda, scaler, alpha=0.25):
    """
    feature_df: same shape/features as in train_risk_model()
    lda, scaler: from train_risk_model()
    alpha: how aggressively we discount high-risk skins (0.25 => up to 25%)
    Returns: report_df with all pricing info
    """

    # Select same features used for training
    X_all = feature_df[[
        "mean_price_usd",
        "mean_pct_change",
        "volatility",
        "mean_deviation",
        "spike_count"
    ]].copy()

    # --- 🧹 Clean data before scaling ---
    import numpy as np
    X_all = X_all.replace([np.inf, -np.inf], np.nan)
    X_all = X_all.fillna(0)
    X_all = np.clip(X_all, -1e6, 1e6) 
    X_all = np.array(X_all, dtype=float)

    # --- Scale features ---
    X_all_scaled = scaler.transform(X_all)

    # --- Predict risk probability ---
    if hasattr(lda, "predict_proba"):
        risk_prob = lda.predict_proba(X_all_scaled)[:, 1]
    else:
        risk_pred = lda.predict(X_all_scaled)
        risk_prob = risk_pred.astype(float)

    # --- Compute safe price ---
    safe_price = feature_df["mean_price_usd"].values * (1 - alpha * risk_prob)

    report_df_LDA = pd.DataFrame({
        "market_hash_name": feature_df.index,
        "mean_price_usd": feature_df["mean_price_usd"].values,
        "volatility": feature_df["volatility"].values,
        "spike_count": feature_df["spike_count"].values,
        "risk_score_model": risk_prob,
        "safe_price_usd": safe_price
    })

    report_df_LDA = report_df_LDA.sort_values(by="risk_score_model", ascending=False)
    return report_df_LDA

In [4]:
lda, scaler, labeled_feature_df = train_risk_model(feature_df)
report_df_LDA = generate_client_report(labeled_feature_df, lda, scaler, alpha=0.25)
print(report_df_LDA.head(10))

[LDA] Training accuracy: 0.938
[LDA] Testing  accuracy: 0.945
Class balance (0 normal / 1 risky):
0    22852
1     3983
Name: count, dtype: int64
                                        market_hash_name  mean_price_usd  \
12192            StatTrak™ P250 | Valence (Minimal Wear)        0.566667   
8679   Souvenir Charm | Austin 2025 Highlight | nqz A...       23.745455   
15365      Sticker | Into The Breach (Holo) | Paris 2023        0.733333   
9534   Souvenir PP-Bizon | Facility Sketch (Factory New)        0.411111   
762    Autograph Capsule | Counter Logic Gaming | Col...      373.402985   
12395      StatTrak™ PP-Bizon | Space Cat (Minimal Wear)        0.655556   
20768              Sticker | qikert (Glitter) | Rio 2022        0.511111   
13766                Sticker | Buzz (Holo) | Austin 2025        0.227273   
9050            Souvenir M4A4 | Mainframe (Minimal Wear)        0.333333   
22458             XM1014 | Blaze Orange (Battle-Scarred)     1783.000000   

       volatility